# Golden Build (GPT-5.5) — CON / PEP / RPT

## 공통

In [28]:
!pip install -q openai
import json, time
from collections import Counter
from openai import OpenAI
from google.colab import userdata
from tqdm.auto import tqdm

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
MODEL = 'gpt-5.5'
REASONING = 'medium'

In [29]:
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def get_nested(d, path):
    cur = d
    for k in path.split('.'):
        cur = cur.get(k) if isinstance(cur, dict) else None
    return cur

def parse_arr(txt):
    if not txt: return []
    p = json.loads(txt)
    if isinstance(p, list): return p
    for v in p.values():
        if isinstance(v, list): return v
    return [p]

def cohen_kappa(g, pr, labels):
    ids = [i for i in g if i in pr]; n = len(ids)
    if n == 0: return None, 0.0, 0
    po = sum(1 for i in ids if g[i]==pr[i]) / n
    pe = 0.0
    for L in labels:
        pe += (sum(1 for i in ids if g[i]==L)/n) * (sum(1 for i in ids if pr[i]==L)/n)
    return ((po-pe)/(1-pe) if pe < 1 else 1.0), po, n

def interp(k):
    if k is None: return 'N/A'
    return ('almost-perfect(>=0.8)' if k>0.8 else 'substantial(0.6-0.8)' if k>0.6
            else 'moderate(0.4-0.6)' if k>0.4 else 'low(<0.4)')

def confusion(g, pr, labels):
    m = {a:{b:0 for b in labels} for a in labels}
    for i in g:
        if i in pr and g[i] in labels and pr[i] in labels:
            m[g[i]][pr[i]] += 1
    return m

def _instruction(labels, boundary, criteria_rules):
    lab = '|'.join(labels)
    return (
        '너는 공공 SW 문서 검토자다. 각 건의 입력만 보고 아래 기준으로 판정하라.\n'
        f'- 판정값은 {labels} 중 하나다.\n'
        '- 근거는 주어진 입력 안에만 둔다. 외부 지식으로 없는 기준을 채우지 마라.\n'
        f'- {boundary}\n'
        f'{criteria_rules}\n'
        '- 입력에 정답 label은 없다. 위 기준으로 독립 판정한다.\n'
        f'출력은 JSON 배열만: [{{"id":"...","판정":"{lab}","이유":"한 문장 근거"}}]'
    )

def judge_gpt55(payload, instruction, retries=3):
    for a in range(retries):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{'role':'system','content':instruction},
                          {'role':'user','content':json.dumps(payload, ensure_ascii=False)}],
                response_format={'type':'json_object'},
                reasoning_effort=REASONING,
                max_completion_tokens=16000)
            txt = r.choices[0].message.content
            if not txt:
                print('  empty(finish=%s) retry' % r.choices[0].finish_reason); time.sleep(3); continue
            return parse_arr(txt)
        except Exception as e:
            print('  err:', e); time.sleep(5)
    return []

def run_iaa(data, gold_field, labels, input_fields, boundary, criteria_rules, task_name, batch=1):
    gold = {r['id']: (get_nested(r, gold_field) if '.' in gold_field else r.get(gold_field)) for r in data}
    instr = _instruction(labels, boundary, criteria_rules)
    pred = {}
    reasons = {}                              # ← 추가
    for b in tqdm(list(chunks(data, batch)), desc=f'{task_name} GPT-5.5', unit='batch'):
        payload = []
        for it in b:
            item = {'id': it['id']}
            for f in input_fields:
                item[f.split('.')[-1]] = get_nested(it, f) if '.' in f else it.get(f, '')
            payload.append(item)
        for v in judge_gpt55(payload, instr):
            if 'id' in v and v.get('판정') in labels:
                pred[v['id']] = v['판정']
                reasons[v['id']] = v.get('이유', '')   # ← 추가
        time.sleep(2)
    k, po, n = cohen_kappa(gold, pred, labels)
    print(f'\n== [{task_name}] IAA my-gold vs GPT-5.5 (n={n}) ==')
    print(f'  accuracy={po:.3f}  kappa={k:.3f} -> {interp(k)}')
    m = confusion(gold, pred, labels)
    print('        ' + ''.join(f'{l:>6}' for l in labels))
    for g in labels:
        print(f'  {g:>4} ' + ''.join(f'{m[g][p]:>6}' for p in labels))
    diff = [(i, gold[i], pred[i]) for i in gold if i in pred and gold[i]!=pred[i]]
    print(f'  mismatch {len(diff)}:')
    for i, g, p in diff:
        print(f'    {i}: my={g} / gpt={p}  ← 이유: {reasons.get(i,"")}')   # ← 이유 출력
    # mismatch에 이유 포함해서 저장
    json.dump([{'id':i,'my_gold':g,'gpt55':p,'gpt_이유':reasons.get(i,'')} for i,g,p in diff],
              open(f'{task_name}_golden_review.json','w'), ensure_ascii=False, indent=2)
    # 전체 판정+이유도 따로 저장 (원하면)
    json.dump({i:{'판정':pred[i],'이유':reasons.get(i,'')} for i in pred},
              open(f'{task_name}_gpt55_all.json','w'), ensure_ascii=False, indent=2)
    print(f'  -> {task_name}_golden_review.json (mismatch+이유), {task_name}_gpt55_all.json (전체+이유) saved')
    return gold, pred, diff

In [37]:
CON_RULES = """[판정 기준]
- 일치: clause_text의 핵심 수치·조건이 refs.chunk_text 기준과 동일하고, refs에 비교 가능한 기준이 실제로 존재한다.
- 일치경계(B) → 판정은 '일치': 핵심 수치·조건(율·기한·비율 등)은 refs와 같지만, refs가 예외·단서·세부조건(재난 시 단축, 경제위기 시 완화, 기산 방식, 절차 등)까지 함께 규정하는데 clause_text가 그 예외·세부를 생략하고 원칙만 규정한 경우. 이때 원칙 수치가 refs와 같으면 '일치'로 본다. 예외·세부 생략만을 이유로 '불일치'로 바꾸지 마라.
- 불일치: clause_text의 핵심 수치·조건 자체가 refs 기준과 다르다(초과·미달·다른 값·다른 산정기준·배제·면제). 예외 생략이 아니라 '핵심 값이나 산정 기준이 refs와 어긋날 때'만 불일치다.
- 검토(RAG miss): 쟁점을 판단할 decisive 기준이 refs에 없다. refs가 같은 주제로 보여도 핵심 수치·조건을 직접 비교할 기준(예: 해당 계약유형의 율, 처리기한)이 refs에 없으면 검토. 외부 지식으로 정답 조문을 상정하지 않는다.

[판정 순서]
1) clause의 핵심 수치·조건을 refs에서 직접 비교할 기준이 있는가? 없으면 → 검토.
2) 있으면, 핵심 값이 같은가? 같으면 → 일치(예외·세부가 refs에만 더 있어도 일치 유지 = 일치경계).
3) 핵심 값 자체가 다르거나 산정 기준이 다르면 → 불일치."""

PEP_RULES = """[특성별 판정 기준]
- 완전성: RFP 요구와 PEP 대응이 같은 산출물·같은 담당주체·같은 시점으로 읽히면 충족 / 요구 항목이 유사성조차 없거나 조건부 문구면 불가 / 동일 개념인지 애매하면 검토.
- 정확성: 수치·규격 정확 일치 충족 / 수치 불일치·PEP 내부 자기모순·비교 기준값 부재 불가 / 검토 없음.
- 검증가능성: 숫자·기한·비율·건수·도구명+기준 있으면 충족(블랙리스트 있어도 제거 후 독립검증 가능하면 충족) / 검증 기준 전무하면 불가 / 블랙리스트('적절히','최선을 다해','원활히 협의하여','필요한 범위에서','우수한 수준으로','성실히','무리 없이','신속히') 있고 유추 애매하면 검토.
- 추적성: RFP 과업·단계·산출물·일정·목표가 PEP에서 1:1 대응 충족 / 대응 항목 없음 불가 / 명칭 달라 애매하면 검토.
[종합] 불가 1개 이상이면 불가 / 불가 없고 검토 1개 이상이면 검토 / 전부 충족이면 충족."""

RPT_RULES = """[특성별 판정 기준]
- 완전성: PEP 과업·범위·산출물이 RPT에 모두 대응(동의어 인정, 같은 산출물·주체·시점 명백)하면 충족 / 하나라도 유사성조차 없거나 조건부('필요 시','~할 수 있다','추후 검토')면 불가 / 같은 산출물·주체·시점인지 갈리면 검토.
- 정확성: PEP 계획값과 RPT 결과값 직접 대조 일치 충족 / PEP값과 불일치·RPT 내부 자기모순·비교 기준값 부재 불가 / 검토 없음.
- 검증가능성: 수치·건수·기준일·비율·시험결과·검수결과·측정값으로 확인가능 충족(블랙리스트 있어도 제거 후 독립검증 가능하면 충족) / 검증 서술 없거나 블랙리스트만 있으면 불가 / 블랙리스트('적절히','최선을 다해','원활히 협의하여','필요한 범위에서','우수한 수준으로','성실히','무리 없이','신속히') 있고 유추 애매하면 검토.
- 추적성: PEP 단계·산출물과 RPT 결과 1:1 명확 대응 충족(보조신호 없어도 명확하면 충족) / 대응 안 되고 보조신호 3개(산출물명겹침·순번개수대응·기능설명일치) 모두 없으면 불가 / 재구성돼 보조신호 일부만 있어 1:1 확정 어려우면 검토.
[종합] 불가 1개 이상이면 불가 / 불가 없고 검토 1개 이상이면 검토 / 전부 충족이면 충족."""

## 계약서

In [38]:
raw = json.load(open('con_seed_v3.json', encoding='utf-8'))
con_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
print('CON', len(con_data), dict(Counter(r.get('label') for r in con_data)))
con_gold, con_pred, con_diff = run_iaa(
    con_data, 'label', ['일치','불일치','검토'],
    ['clause_text','refs'],
    "refs에 비교 기준 없으면 '검토'(RAG miss).",
    CON_RULES, 'CON', batch=1)

CON 50 {'일치': 17, '불일치': 23, '검토': 10}


CON GPT-5.5:   0%|          | 0/50 [00:00<?, ?batch/s]


== [CON] IAA my-gold vs GPT-5.5 (n=50) ==
  accuracy=0.880  kappa=0.811 -> almost-perfect(>=0.8)
            일치   불일치    검토
    일치     12     1     4
   불일치      0    23     0
    검토      0     1     9
  mismatch 6:
    CON_013: my=일치 / gpt=불일치  ← 이유: refs는 시정 완료 통지를 받은 날부터 검사기간을 다시 계산한다고 규정하지만, clause_text는 보완 소요 기간만 검사기간에 산입하지 않는다고 하여 검사기간 산정 기준이 다르다.
    CON_022: my=일치 / gpt=검토  ← 이유: refs에는 계약보증금이 계약금액의 100분의 10 이상이라는 기준은 있으나, clause_text의 보증기간을 이행기간에 30일 가산한다는 조건을 직접 비교할 기준이 없다.
    CON_026: my=일치 / gpt=검토  ← 이유: refs에는 이행 촉구에 따르지 않는 경우 해지할 수 있다는 일반 사유만 있고, clause_text의 ‘착수하지 아니한 때 14일 이상의 기간을 정하여 최고’라는 핵심 기한·조건을 직접 비교할 기준이 없다.
    CON_040: my=일치 / gpt=검토  ← 이유: refs는 설계변경에 따른 계약금액 조정 기준을 규정하지만, clause_text의 ‘경미한 과업 조정은 통지로 반영하고 대가는 별도로 정한다’는 조건을 직접 비교할 결정적 기준은 제시하지 않는다.
    CON_041: my=일치 / gpt=검토  ← 이유: refs에는 공동 창작 계약의 저작권 공동 소유 및 계약상대자의 산출물 반출 승인 기준은 있으나, clause_text처럼 산출물 저작권을 일반적으로 발주기관과 계약상대자에게 공동 귀속시키고 발주기관의 활용을 보장한다는 기준을 직접 비교할 결정적 기준이 없다.
    CON_043: my=검토 / gpt=불일치  ←

## 과업수행계획서

In [ ]:
raw = json.load(open('pep_seed_golden.json', encoding='utf-8'))
pep_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
print('PEP', len(pep_data), dict(Counter(get_nested(r,'output.label') for r in pep_data)))
pep_gold, pep_pred, pep_diff = run_iaa(
    pep_data, 'output.label', ['충족','불가','검토'],
    ['input.criteria','input.rfp_excerpt','input.pep_excerpt'],
    "criteria 특성만 보고 대로 확정 안 되면 '검토'. 정확성은 검토 금지.",
    PEP_RULES, 'PEP', batch=1)

## 결과보고서

In [ ]:
raw = json.load(open('pep_seed_golden.json', encoding='utf-8'))
pep_data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw
print('PEP', len(pep_data), dict(Counter(get_nested(r,'output.label') for r in pep_data)))
pep_gold, pep_pred, pep_diff = run_iaa(
    pep_data, 'output.label', ['충족','불가','검토'],
    ['input.criteria','input.rfp_excerpt','input.pep_excerpt'],
    "criteria 특성만 보고 대조로 확정 안 되면 '검토'. 정확성은 검토 금지.",
    PEP_RULES, 'PEP', batch=1)